# 05 — Avaliação Final dos Modelos

Este notebook avalia os modelos treinados no conjunto de teste,
gera métricas finais, tabelas LaTeX e gráficos de avaliação.

**Pré-requisito:** Execute `04_treinamento_modelo.ipynb` antes.

---

## 5.1 Importações

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import (
    roc_auc_score, f1_score, accuracy_score,
    confusion_matrix, classification_report, roc_curve,
)

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})
sns.set_theme(style="whitegrid")

CAMINHO_TESTE = os.path.join("..", "dados", "processado", "dataset_teste.csv")
DIR_MODELOS = os.path.join("..", "modelos")
DIR_FIGURAS = os.path.join("..", "resultados", "figuras")
DIR_METRICAS = os.path.join("..", "resultados", "metricas")
os.makedirs(DIR_FIGURAS, exist_ok=True)
os.makedirs(DIR_METRICAS, exist_ok=True)

## 5.2 Carregar Dados e Modelos

In [ ]:
# Dados de teste
df = pd.read_csv(CAMINHO_TESTE)
X = df.drop(columns=["churn"])
y = df["churn"]
print(f"Teste: {X.shape[0]} registros, {X.shape[1]} features")

# Modelos
modelos = {}
for arq in sorted(os.listdir(DIR_MODELOS)):
    if arq.endswith(".joblib"):
        nome = arq.replace(".joblib", "").replace("_", " ").title()
        modelos[nome] = joblib.load(os.path.join(DIR_MODELOS, arq))
        print(f"  ✓ {nome}")

## 5.3 Métricas no Conjunto de Teste

In [ ]:
resultados = []

for nome, modelo in modelos.items():
    y_pred = modelo.predict(X)
    y_proba = modelo.predict_proba(X)[:, 1]
    
    resultado = {
        "Modelo": nome,
        "AUC-ROC": roc_auc_score(y, y_proba),
        "F1-Score": f1_score(y, y_pred),
        "Acurácia": accuracy_score(y, y_pred),
    }
    resultados.append(resultado)

df_metricas = pd.DataFrame(resultados).sort_values("AUC-ROC", ascending=False)
df_metricas

## 5.4 Curva ROC

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
cores = ["#2ecc71", "#3498db", "#e74c3c", "#9b59b6"]

for (nome, modelo), cor in zip(modelos.items(), cores):
    y_proba = modelo.predict_proba(X)[:, 1]
    fpr, tpr, _ = roc_curve(y, y_proba)
    auc = roc_auc_score(y, y_proba)
    ax.plot(fpr, tpr, color=cor, lw=2, label=f"{nome} (AUC = {auc:.3f})")

ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5, label="Aleatório (AUC = 0.500)")
ax.set_xlabel("Taxa de Falsos Positivos (FPR)")
ax.set_ylabel("Taxa de Verdadeiros Positivos (TPR)")
ax.set_title("Curva ROC — Comparação de Modelos", fontsize=14, fontweight="bold")
ax.legend(loc="lower right", fontsize=10)
ax.grid(True, alpha=0.3)

fig.savefig(os.path.join(DIR_FIGURAS, "curva_roc.png"), dpi=150, bbox_inches="tight")
plt.show()

## 5.5 Importância de Features

In [ ]:
# Encontrar modelo com feature_importances_
for nome, modelo in modelos.items():
    if hasattr(modelo, "feature_importances_"):
        importancias = pd.Series(modelo.feature_importances_, index=X.columns).sort_values(ascending=True)
        top = importancias.tail(15)
        
        fig, ax = plt.subplots(figsize=(10, 7))
        ax.barh(top.index, top.values, color="#3498db", edgecolor="white")
        ax.set_xlabel("Importância")
        ax.set_title(f"Top 15 Features — {nome}", fontsize=14, fontweight="bold")
        
        fig.savefig(os.path.join(DIR_FIGURAS, "importancia_features.png"), dpi=150, bbox_inches="tight")
        plt.show()
        break

## 5.6 Matriz de Confusão (Melhor Modelo)

In [ ]:
# Selecionar melhor modelo por AUC
melhor = df_metricas.iloc[0]
melhor_modelo = modelos[melhor["Modelo"]]
y_pred = melhor_modelo.predict(X)

print(f"Melhor modelo: {melhor['Modelo']}\n")
print(classification_report(y, y_pred, target_names=["Não-Churn", "Churn"]))

cm = confusion_matrix(y, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Não-Churn", "Churn"],
            yticklabels=["Não-Churn", "Churn"], ax=ax)
ax.set_xlabel("Predito")
ax.set_ylabel("Real")
ax.set_title(f"Matriz de Confusão — {melhor['Modelo']}", fontweight="bold")
plt.tight_layout()
plt.show()

## 5.7 Salvar Resultados

In [ ]:
df_metricas.to_csv(os.path.join(DIR_METRICAS, "metricas_teste.csv"), index=False)
print("✓ Métricas salvas em resultados/metricas/metricas_teste.csv")
print("✓ Figuras salvas em resultados/figuras/")